# গড় আয়ু বিশ্লেষণ

বিশ্বব্যাপী মানুষের গড় আয়ু অন্বেষণকারী দুটি ডেটাসেট:
- **Gapminder** (1952–2007): country, year, population, continent, lifeExp, gdpPercap
- **WHO গড় আয়ু** (2000–2015): ১৯৩টি দেশ, ২২টি নির্দেশক (মৃত্যুহার, বিএমআই/BMI, জিডিপি, শিক্ষা ইত্যাদি)

এই ওয়ার্কবুকটি **Python** এবং **R** উভয় ভাষায় CSV ফাইল ইমপোর্ট এবং ডেটা বিশ্লেষণ প্রদর্শন করে।

## ১. সেটআপ: প্যাকেজ ইনস্টল করুন এবং ডেটাসেট ডাউনলোড করুন

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('pandas + plotly ইনস্টল করা হয়েছে')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"ইতিমধ্যেই বিদ্যমান: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"ডাউনলোড করা হয়েছে {name}: {lines} লাইন")

## ২. Gapminder: Python দিয়ে অন্বেষণ

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"আকার: {gap.shape}")
print(f"মহাদেশসমূহ: {sorted(gap['continent'].unique())}")
print(f"বছরের পরিসর: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# মহাদেশ অনুযায়ী সময়ের সাথে গড় আয়ু
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='মহাদেশ অনুযায়ী গড় আয়ু (1952-2007)',
              labels={'lifeExp': 'গড় আয়ু (বছর)', 'year': 'বছর'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# জিডিপি বনাম গড় আয়ু (2007), বাবল সাইজ = জনসংখ্যা
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='জিডিপি বনাম গড় আয়ু (2007)',
                 labels={'gdpPercap': 'মাথাপিছু জিডিপি (log)', 'lifeExp': 'গড় আয়ু'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## ৩. Gapminder: R দিয়ে অন্বেষণ

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# মহাদেশ অনুযায়ী গড় আয়ুর বণ্টন (বক্সপ্লট)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "মহাদেশ অনুযায়ী গড় আয়ু",
        xlab = "মহাদেশ", ylab = "গড় আয়ু (বছর)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# গড় আয়ু বৃদ্ধির শীর্ষে থাকা ১০টি দেশ (1952 বনাম 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "শীর্ষ ১০: গড় আয়ু বৃদ্ধি (1952-2007)",
        xlab = "বৃদ্ধি পাওয়া বছর",
        col = "#00CC96", border = NA)

## ৪. WHO গড় আয়ু: Python দিয়ে অন্বেষণ

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"আকার: {who.shape}")
print(f"কলামসমূহ: {list(who.columns)}")
print(f"\nঅনুপস্থিত মান (শীর্ষ ৫):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# উন্নয়নশীল বনাম উন্নত: প্রাক-বিনকৃত গড় আয়ুর বণ্টন
# সুনির্দিষ্ট বার স্থানাঙ্ক ব্রাউজার Plotly ব্রিজের মাধ্যমে সঠিকভাবে রেন্ডার হয়।
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='গড় আয়ু: উন্নয়নশীল বনাম উন্নত',
             labels={'Life expectancy': 'গড় আয়ু (বছর)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# শিক্ষা বনাম গড় আয়ু
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='শিক্ষা বনাম গড় আয়ু (2014)',
                 labels={'Life expectancy': 'গড় আয়ু (বছর)',
                         'Schooling': 'শিক্ষার বছর'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## ৫. WHO গড় আয়ু: R দিয়ে অন্বেষণ

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nদেশসমূহ:", length(unique(who$Country)))
cat("\nবছরের পরিসর:", range(who$Year))

In [ ]:
# সহসম্পর্ক: প্রাপ্তবয়স্কদের মৃত্যুহার বনাম গড় আয়ু
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "প্রাপ্তবয়স্কদের মৃত্যুহার বনাম গড় আয়ু",
     xlab = "প্রাপ্তবয়স্কদের মৃত্যুহার (প্রতি ১০০০ জনে)",
     ylab = "গড় আয়ু (বছর)")
legend("topright", legend = c("উন্নত", "উন্নয়নশীল"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# সরল রৈখিক মডেল: গড় আয়ুর পূর্বাভাস দেয় কোনগুলো?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## মূল সিদ্ধান্তসমূহ

- বিশ্বব্যাপী গড় আয়ু বৃদ্ধি পেয়েছে, তবে মহাদেশগুলোর মধ্যে এখনও বড় ব্যবধান রয়েছে
- জিডিপি এবং শিক্ষা গড় আয়ু বৃদ্ধির শক্তিশালী ইতিবাচক নির্দেশক
- প্রাপ্তবয়স্কদের মৃত্যুহার সবচেয়ে শক্তিশালী নেতিবাচক নির্দেশক
- উন্নয়নশীল দেশগুলোতে ফলাফলের ক্ষেত্রে অনেক বড় বৈচিত্র্য (variance) দেখা যায়